# Pandas Business Practice — 20 solved problems on order data

01 Core Python · **▶ 02 Pandas** · 03 Cleaning · 04 Transformation · 05 Feature Engineering · 06 Regression · 07 Model Prep · 08 Case Studies

`02_Pandas_Essentials/07_pandas_business_practice_solved.ipynb`

---

### In one paragraph (no jargon)

Twenty short problems on a small orders table, each one a pattern you'll meet again. They're worth working through in order because they build: select, filter, add a column, group, merge, reshape, then date handling. The data is created in the setup cell, so you can change a number and re-run any problem to see what happens — which is the fastest way to make these stick.

### After this notebook you can

- Answer the standard select / filter / derive / group questions on a real-ish orders table
- Merge orders with a customer lookup and aggregate the result
- Work with dates: extract the month, group by period, compute a running total
- Recognise, from the wording alone, which pandas method a question is asking for

**Assumed knowledge:** notebooks 01–04 of this section

### What's inside

1. Setup — the sample business data
2. Q1–Q5 · selecting, filtering, deriving
3. Q6–Q10 · grouping and aggregating
4. Q11–Q15 · merging and reshaping
5. Q16–Q20 · dates, ranking and text
6. ⚡ The same twenty answers as one chained pipeline
7. Exam quick-reference

---

> **▶ Runs on its own.** The next cell is the only setup you need. It imports the
> libraries and loads the data. If the `datasets/` folder isn't where it expects,
> it rebuilds an equivalent dataset in memory so **every cell below still runs** —
> handy if you copy this single `.ipynb` somewhere else.
>
> **▶ Reading the cells.** Code comments explain *what the line does*; the text
> blocks explain *why you'd do it*. Look for these markers:
> `# WHAT:` a plain-English translation · `# WHY:` the reason it matters ·
> `# 🔧 CHANGE THIS:` the knob to turn when the exam question differs ·
> **⚡ Beyond the syllabus** = optional, higher-mark techniques.

In [1]:
# =============================================================================
# SETUP — run this cell first. It is the only cell with dependencies.
# =============================================================================
# WHAT: `import` pulls in code other people have written so we don't rewrite it.
#       The `as pd` part is a nickname, so we can type `pd` instead of `pandas`.
import pandas as pd          # tables of data (think: Excel, but programmable)
import numpy as np           # fast maths on whole columns at once
import matplotlib.pyplot as plt   # charts
import warnings

warnings.filterwarnings('ignore')          # hide version-upgrade notices, keeps output readable
pd.set_option('display.max_columns', 50)   # don't hide columns behind "..."
pd.set_option('display.width', 160)
import os

def find_datasets_folder(start=None):
    """Walk upwards from this notebook looking for the shared `datasets/` folder.

    WHY: it means the notebook works whether you opened it from its own folder,
    from the top of the notes, or from anywhere else on your machine.
    """
    here = os.path.abspath(start or os.getcwd())
    for _ in range(6):                       # look up to 6 folders up
        candidate = os.path.join(here, 'datasets')
        if os.path.isdir(candidate):
            return candidate
        parent = os.path.dirname(here)
        if parent == here:
            break
        here = parent
    return None

def dataset_path(filename, rebuild=None):
    """Return a real path to `filename`, materialising a temp copy if it's missing.

    WHY: a few pandas tools (pd.ExcelFile, pd.read_sql) need an actual file path
         rather than a DataFrame, so the fallback has to be written to disk.
    """
    folder = find_datasets_folder()
    if folder:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    if rebuild is None:
        raise FileNotFoundError(filename)
    import tempfile
    tmp = os.path.join(tempfile.mkdtemp(prefix='bda_'), filename)
    frame = rebuild()
    (frame.to_excel(tmp, index=False) if filename.lower().endswith(('.xlsx', '.xls'))
     else frame.to_csv(tmp, index=False))
    print(f"'{filename}' not found -> wrote a rebuilt copy to {tmp}")
    return tmp

def load_data(filename, rebuild=None, **read_kwargs):
    """Load `filename` from the shared datasets folder, or rebuild it in memory.

    WHAT: tries to read the real file; if it can't find it, calls `rebuild()`
          which recreates a dataset with the same columns and behaviour.
    WHY:  guarantees this notebook runs even if the CSV goes missing.
    """
    folder = find_datasets_folder()
    if folder:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            reader = pd.read_excel if filename.lower().endswith(('.xlsx', '.xls')) else pd.read_csv
            print(f"Loaded '{filename}' from {folder}")
            return reader(path, **read_kwargs)
    if rebuild is None:
        raise FileNotFoundError(f"Could not find {filename} and no fallback was supplied.")
    print(f"'{filename}' not found on disk -> rebuilding an equivalent dataset in memory.")
    built = rebuild()
    if 'chunksize' in read_kwargs:            # keep chunked reads working on the fallback path
        size = read_kwargs['chunksize']
        return (built.iloc[i:i + size] for i in range(0, len(built), size))
    return built

print("Setup complete. pandas", pd.__version__, "| numpy", np.__version__)

Setup complete. pandas 3.0.2 | numpy 2.4.4


# Pandas Business Practice — **Solutions**
Solutions for the 20 practice problems. Each cell is self‑contained and runnable.

## 0) Setup: Create Sample Business Data

In [2]:
import pandas as pd
import numpy as np
np.random.seed(7)

orders = pd.DataFrame({
    'order_id': [101,102,103,104,105,106,107,108],
    'order_date': pd.to_datetime([
        '2025-01-02','2025-01-03','2025-01-03','2025-01-07',
        '2025-02-01','2025-02-03','2025-02-10','2025-03-01'
    ]),
    'customer_id': [1,2,1,3,2,4,3,5],
    'product_id': ['P01','P02','P03','P01','P02','P03','P01','P04'],
    'qty': [2,1,3,1,2,4,5,2],
    'unit_price': [500,1200,300,500,1200,300,520,1500],
    'city': ['Delhi','Mumbai','Delhi','Pune','Mumbai','Delhi','Pune','Delhi']
})
orders['total'] = orders['qty'] * orders['unit_price']

customers = pd.DataFrame({
    'customer_id': [1,2,3,4,5],
    'customer_name': ['Ravi','Meera','John','Anita','Kiran'],
    'segment': ['Retail','Corporate','Retail','SMB','Corporate'],
    'signup_date': pd.to_datetime(['2024-12-20','2024-12-22','2025-01-01','2025-01-15','2025-02-01'])
})

products = pd.DataFrame({
    'product_id': ['P01','P02','P03','P04'],
    'product_name': ['Phone','Laptop','Mouse','Printer'],
    'category': ['Electronics','Electronics','Accessories','Peripherals']
})
orders, customers, products

(   order_id order_date  customer_id product_id  qty  unit_price    city  total
 0       101 2025-01-02            1        P01    2         500   Delhi   1000
 1       102 2025-01-03            2        P02    1        1200  Mumbai   1200
 2       103 2025-01-03            1        P03    3         300   Delhi    900
 3       104 2025-01-07            3        P01    1         500    Pune    500
 4       105 2025-02-01            2        P02    2        1200  Mumbai   2400
 5       106 2025-02-03            4        P03    4         300   Delhi   1200
 6       107 2025-02-10            3        P01    5         520    Pune   2600
 7       108 2025-03-01            5        P04    2        1500   Delhi   3000,
    customer_id customer_name    segment signup_date
 0            1          Ravi     Retail  2024-12-20
 1            2         Meera  Corporate  2024-12-22
 2            3          John     Retail  2025-01-01
 3            4         Anita        SMB  2025-01-15
 4            

**Q1. Show the first 3 rows of `orders` and display its columns and dtypes??**

In [3]:
orders.head(3), list(orders.columns), orders.dtypes

(   order_id order_date  customer_id product_id  qty  unit_price    city  total
 0       101 2025-01-02            1        P01    2         500   Delhi   1000
 1       102 2025-01-03            2        P02    1        1200  Mumbai   1200
 2       103 2025-01-03            1        P03    3         300   Delhi    900,
 ['order_id',
  'order_date',
  'customer_id',
  'product_id',
  'qty',
  'unit_price',
  'city',
  'total'],
 order_id                int64
 order_date     datetime64[us]
 customer_id             int64
 product_id                str
 qty                     int64
 unit_price              int64
 city                      str
 total                   int64
 dtype: object)

**Q2. Select only the `order_id`, `customer_id`, `qty`, and `total` columns from `orders`??**

In [4]:
orders[['order_id','customer_id','qty','total']]

,order_id,customer_id,qty,total
0,101,1,2,1000
1,102,2,1,1200
2,103,1,3,900
3,104,3,1,500
4,105,2,2,2400
5,106,4,4,1200
6,107,3,5,2600
7,108,5,2,3000


**Q3. Filter all Delhi orders with `total >= 1000` and show `order_id, city, total`??**

In [5]:
orders.loc[(orders['city']=='Delhi') & (orders['total']>=1000), ['order_id','city','total']]

,order_id,city,total
0,101,Delhi,1000
5,106,Delhi,1200
7,108,Delhi,3000


**Q4. Add `discounted_total = total * 0.9`. Show top 5 by `discounted_total` desc??**

In [6]:
orders2 = orders.copy()
orders2['discounted_total'] = orders2['total'] * 0.9
orders2.sort_values('discounted_total', ascending=False).head(5)

,order_id,order_date,customer_id,product_id,qty,unit_price,city,total,discounted_total
7,108,2025-03-01,5,P04,2,1500,Delhi,3000,2700.0
6,107,2025-02-10,3,P01,5,520,Pune,2600,2340.0
4,105,2025-02-01,2,P02,2,1200,Mumbai,2400,2160.0
1,102,2025-01-03,2,P02,1,1200,Mumbai,1200,1080.0
5,106,2025-02-03,4,P03,4,300,Delhi,1200,1080.0


**Q5. Create `high_value` = 'Yes' if `total >= 1500` else 'No'. Show counts??**

In [7]:
orders3 = orders.copy()
orders3['high_value'] = np.where(orders3['total']>=1500, 'Yes', 'No')
orders3['high_value'].value_counts()

high_value
No     5
Yes    3
Name: count, dtype: int64

**Q6. Compute sum of total by `city` and sort descending??**

In [8]:
orders.groupby('city')['total'].sum().sort_values(ascending=False)

city
Delhi     6100
Mumbai    3600
Pune      3100
Name: total, dtype: int64

**Q7. Compute mean qty and mean total by `product_id` and include `product_name`??**

In [9]:
prod_stats = orders.groupby('product_id').agg(mean_qty=('qty','mean'), mean_total=('total','mean')).reset_index()
prod_stats.merge(products[['product_id','product_name']], on='product_id', how='left')

,product_id,mean_qty,mean_total,product_name
0,P01,2.666667,1366.666667,Phone
1,P02,1.500000,1800.000000,Laptop
2,P03,3.500000,1050.000000,Mouse
3,P04,2.000000,3000.000000,Printer


**Q8. Create `order_month` (YYYY‑MM). Compute monthly revenue (sum of total)??**

In [10]:
o = orders.copy()
o['order_month'] = o['order_date'].dt.to_period('M').astype(str)
o.groupby('order_month')['total'].sum().reset_index().sort_values('order_month')

,order_month,total
0,2025-01,3600
1,2025-02,6200
2,2025-03,3000


**Q9. Number of unique customers per month (`order_month`)??**

In [11]:
o = orders.copy()
o['order_month'] = o['order_date'].dt.to_period('M').astype(str)
o.groupby('order_month')['customer_id'].nunique().reset_index(name='unique_customers')

,order_month,unique_customers
0,2025-01,3
1,2025-02,3
2,2025-03,1


**Q10. Left‑join `orders` with `customers` (bring name & segment). Show first 6 rows??**

In [12]:
oc = orders.merge(customers, on='customer_id', how='left')
oc.head(6)

,order_id,order_date,customer_id,product_id,qty,unit_price,city,total,customer_name,segment,signup_date
0,101,2025-01-02,1,P01,2,500,Delhi,1000,Ravi,Retail,2024-12-20
1,102,2025-01-03,2,P02,1,1200,Mumbai,1200,Meera,Corporate,2024-12-22
2,103,2025-01-03,1,P03,3,300,Delhi,900,Ravi,Retail,2024-12-20
3,104,2025-01-07,3,P01,1,500,Pune,500,John,Retail,2025-01-01
4,105,2025-02-01,2,P02,2,1200,Mumbai,2400,Meera,Corporate,2024-12-22
5,106,2025-02-03,4,P03,4,300,Delhi,1200,Anita,SMB,2025-01-15


**Q11. Compute revenue by segment (sum of total) after the join??**

In [13]:
oc.groupby('segment')['total'].sum().sort_values(ascending=False).reset_index()

,segment,total
0,Corporate,6600
1,Retail,5000
2,SMB,1200


**Q12. Introduce a missing city, fill with 'Unknown', and show counts??**

In [14]:
missing = orders.copy()
missing.loc[missing.index[0], 'city'] = np.nan
missing['city'] = missing['city'].fillna('Unknown')
missing['city'].value_counts(dropna=False)

city
Delhi      3
Mumbai     2
Pune       2
Unknown    1
Name: count, dtype: int64

**Q13. Trim and title‑case `customer_name` (show before vs after)??**

In [15]:
cust_dirty = customers.copy()
cust_dirty.loc[0,'customer_name'] = '  ravi  '
cust_dirty.loc[1,'customer_name'] = 'MEERA'
before_after = pd.DataFrame({
    'before': cust_dirty['customer_name'],
    'after': cust_dirty['customer_name'].str.strip().str.title()
})
before_after

,before,after
0,ravi,Ravi
1,MEERA,Meera
2,John,John
3,Anita,Anita
4,Kiran,Kiran


**Q14. Pivot of sum(total) with rows=city, cols=product_id (fillna=0)??**

In [16]:
pivot = orders.pivot_table(index='city', columns='product_id', values='total', aggfunc='sum', fill_value=0)
pivot

product_id,P01,P02,P03,P04
city,,,,
Delhi,1000,0,2100,3000
Mumbai,0,3600,0,0
Pune,3100,0,0,0


**Q15. Melt `orders` to long with id_vars=['order_id','customer_id'] and value_vars=['qty','unit_price','total']??**

In [17]:
long_orders = orders.melt(id_vars=['order_id','customer_id'], value_vars=['qty','unit_price','total'], var_name='metric', value_name='value')
long_orders.head(10)

,order_id,customer_id,metric,value
0,101,1,qty,2
1,102,2,qty,1
2,103,1,qty,3
3,104,3,qty,1
4,105,2,qty,2
5,106,4,qty,4
6,107,3,qty,5
7,108,5,qty,2
8,101,1,unit_price,500
9,102,2,unit_price,1200


**Q16. Select orders where (city in ['Delhi','Mumbai']) AND qty >= 2??**

In [18]:
orders.loc[orders['city'].isin(['Delhi','Mumbai']) & (orders['qty']>=2)]

,order_id,order_date,customer_id,product_id,qty,unit_price,city,total
0,101,2025-01-02,1,P01,2,500,Delhi,1000
2,103,2025-01-03,1,P03,3,300,Delhi,900
4,105,2025-02-01,2,P02,2,1200,Mumbai,2400
5,106,2025-02-03,4,P03,4,300,Delhi,1200
7,108,2025-03-01,5,P04,2,1500,Delhi,3000


**Q17. From the joined data, select new customers (order within 30 days of signup)??**

In [19]:
oc = orders.merge(customers, on='customer_id', how='left')
delta = (oc['order_date'] - oc['signup_date']).dt.days
new_customers = oc.loc[(delta >= 0) & (delta <= 30), ['order_id','customer_name','order_date','signup_date']]
new_customers.sort_values('order_date')

,order_id,customer_name,order_date,signup_date
0,101,Ravi,2025-01-02,2024-12-20
1,102,Meera,2025-01-03,2024-12-22
2,103,Ravi,2025-01-03,2024-12-20
3,104,John,2025-01-07,2025-01-01
5,106,Anita,2025-02-03,2025-01-15
7,108,Kiran,2025-03-01,2025-02-01


**Q18. Add `city_rank` ranking orders by total within each city (1=highest)??**

In [20]:
ranked = orders.copy()
ranked['city_rank'] = ranked.groupby('city')['total'].rank(ascending=False, method='dense')
ranked[['order_id','city','total','city_rank']].sort_values(['city','city_rank','total'])

,order_id,city,total,city_rank
7,108,Delhi,3000,1.0
5,106,Delhi,1200,2.0
0,101,Delhi,1000,3.0
2,103,Delhi,900,4.0
4,105,Mumbai,2400,1.0
1,102,Mumbai,1200,2.0
6,107,Pune,2600,1.0
3,104,Pune,500,2.0


**Q19. Top 2 products by total revenue overall (with product_name)??**

In [21]:
rev = orders.groupby('product_id')['total'].sum().reset_index(name='revenue')
rev = rev.merge(products[['product_id','product_name']], on='product_id', how='left')
rev.sort_values('revenue', ascending=False).head(2)

,product_id,revenue,product_name
0,P01,4100,Phone
1,P02,3600,Laptop


**Q20. Order summary per customer: total orders, total qty, total revenue (top 5)??**

In [22]:
summary = orders.groupby('customer_id').agg(
    orders_cnt=('order_id','nunique'),
    total_qty=('qty','sum'),
    total_revenue=('total','sum')
).reset_index()
summary = summary.merge(customers[['customer_id','customer_name','segment']], on='customer_id', how='left')
summary.sort_values('total_revenue', ascending=False).head(5)

,customer_id,orders_cnt,total_qty,total_revenue,customer_name,segment
1,2,2,3,3600,Meera,Corporate
2,3,2,6,3100,John,Retail
4,5,1,2,3000,Kiran,Corporate
0,1,2,5,1900,Ravi,Retail
3,4,1,4,1200,Anita,SMB


### ⚡ Beyond the syllabus — the same answers, written as one pipeline

Individually these twenty answers are fine. Strung together they show something more valuable: a **repeatable analysis pipeline**. Nothing is reassigned, nothing is mutated, and you can read the business logic top to bottom. This is what an examiner means by 'well-structured code'.

In [23]:
# A complete monthly performance report, built as one expression
report = (
    orders
    .assign(
        order_month     = lambda d: d['order_date'].dt.to_period('M').astype(str),
        discounted      = lambda d: d['total'] * 0.9,
        value_band      = lambda d: pd.cut(d['total'], [-np.inf, 500, 1500, np.inf],
                                           labels=['Small', 'Medium', 'Large']),
    )
    .groupby(['order_month', 'city'], observed=True)
    .agg(orders_placed = ('order_id', 'count'),
         units         = ('qty', 'sum'),
         revenue       = ('total', 'sum'),
         after_discount= ('discounted', 'sum'),
         avg_order     = ('total', 'mean'))
    .round(2)
    .reset_index()
    .sort_values(['order_month', 'revenue'], ascending=[True, False])
)
display(report)

print("\nAnd the headline numbers a manager would actually ask for:")
print(f"  Total revenue      : {orders['total'].sum():>10,.2f}")
print(f"  Orders             : {len(orders):>10,}")
print(f"  Average order value: {orders['total'].mean():>10,.2f}")
print(f"  Best city          : {orders.groupby('city')['total'].sum().idxmax()}")
print(f"  Busiest month      : {orders['order_date'].dt.to_period('M').value_counts().idxmax()}")

,order_month,city,orders_placed,units,revenue,after_discount,avg_order
0,2025-01,Delhi,2,5,1900,1710.0,950.0
1,2025-01,Mumbai,1,1,1200,1080.0,1200.0
2,2025-01,Pune,1,1,500,450.0,500.0
5,2025-02,Pune,1,5,2600,2340.0,2600.0
4,2025-02,Mumbai,1,2,2400,2160.0,2400.0
3,2025-02,Delhi,1,4,1200,1080.0,1200.0
6,2025-03,Delhi,1,2,3000,2700.0,3000.0



And the headline numbers a manager would actually ask for:
  Total revenue      :  12,800.00
  Orders             :          8
  Average order value:   1,600.00
  Best city          : Delhi
  Busiest month      : 2025-01


In [24]:
# Month-on-month growth — the follow-up question that always comes next
monthly = (orders
           .set_index('order_date')
           .resample('ME')['total']            # 'ME' = month end. 'W' weekly, 'QE' quarterly, 'YE' yearly
           .agg(['count', 'sum'])
           .rename(columns={'count': 'orders', 'sum': 'revenue'}))

monthly['prev_revenue'] = monthly['revenue'].shift(1)              # shift = previous row
monthly['growth_pct']   = monthly['revenue'].pct_change() * 100    # % change vs previous row
monthly['running_total'] = monthly['revenue'].cumsum()             # cumulative

display(monthly.round(2))
print("\n.shift(), .pct_change(), .cumsum() and .resample() are the four time-series verbs.")
print("Between them they answer almost every 'compared with last month' question.")

,orders,revenue,prev_revenue,growth_pct,running_total
order_date,,,,,
2025-01-31,4,3600,NaN,NaN,3600
2025-02-28,3,6200,3600.0,72.22,9800
2025-03-31,1,3000,6200.0,-51.61,12800



.shift(), .pct_change(), .cumsum() and .resample() are the four time-series verbs.
Between them they answer almost every 'compared with last month' question.


---

## Exam quick-reference

| To do this | Write this |
|---|---|
| First rows + types | `df.head(3)`, `df.dtypes` |
| Choose columns | `df[['a','b']]` |
| Filter + choose columns | `df.loc[mask, ['a','b']]` |
| New derived column | `df['x'] = df['y'] * 0.9` |
| Two-way flag | `np.where(cond, 'Yes', 'No')` |
| Total by group | `df.groupby('city')['total'].sum()` |
| Sort a summary | `… .sort_values(ascending=False)` |
| Attach a lookup table | `df.merge(cust, on='customer_id', how='left')` |
| Wide summary grid | `df.pivot_table(values=, index=, columns=)` |
| Month from a date | `df['d'].dt.to_period('M')` |
| Group by month | `df.resample('ME', on='d')['v'].sum()` |
| Previous row | `df['v'].shift(1)` |
| Percent change | `df['v'].pct_change()` |
| Running total | `df['v'].cumsum()` |
| Rank rows | `df['v'].rank(ascending=False)` |

### Adapting this in the exam

- 'Show columns X and Y where Z' → `df.loc[mask, ['X','Y']]`.
- 'Add a column that…' → assignment, `np.where`, or `pd.cut` depending on how many outcomes.
- 'By month' → `.dt.to_period('M')` for a label, `.resample('ME')` for a time series.
- 'Compared with the previous period' → `.shift(1)` or `.pct_change()`.

### Traps that cost marks

- Dates read from a file are **text** until you run `pd.to_datetime`. Until then `.dt` fails and sorting is alphabetical.
- `df.copy()` before adding columns, or you'll modify the original and later cells will disagree with earlier ones.
- `value_counts()` sorts by frequency, not by category name. Add `.sort_index()` if you need alphabetical order.
- `groupby` drops rows where the grouping key is blank unless you pass `dropna=False`.
- `resample` needs a DatetimeIndex — use `.set_index('date')` first, or pass `on='date'`.